In [ ]:
from keras.layers import Conv2D, UpSampling2D, InputLayer
from keras.models import Sequential
from skimage.color import rgb2lab, lab2rgb
from skimage.io import imsave
import numpy as np
from google.colab import files
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

upl = files.upload()
names = list(upl.keys())
img = Image.open(BytesIO(upl[names[0]]))

def processed_image(img):
  image = img.resize( (256, 256), Image.BILINEAR)
  image = np.array(image, dtype=float)
  size = image.shape
  lab = rgb2lab(1.0/255*image)
  X, Y = lab[:,:,0], lab[:,:,1:]

  Y /= 128    # нормируем выходные значение в диапазон от -1 до 1
  X = X.reshape(1, size[0], size[1], 1)
  Y = Y.reshape(1, size[0], size[1], 2)
  return X, Y, size

X, Y, size = processed_image(img)

model = Sequential()
model.add(InputLayer(input_shape=(None, None, 1)))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same', strides=2))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same', strides=2))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same', strides=2))
model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(UpSampling2D((2, 2)))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(UpSampling2D((2, 2)))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(Conv2D(2, (3, 3), activation='tanh', padding='same'))
model.add(UpSampling2D((2, 2)))

model.compile(optimizer='adam', loss='mse')
model.fit(x=X, y=Y, batch_size=1, epochs=50)

upl = files.upload()
names = list(upl.keys())
img = Image.open(BytesIO(upl[names[0]]))
X, Y, size = processed_image(img)

output = model.predict(X)

output *= 128
min_vals, max_vals = -128, 127
ab = np.clip(output[0], min_vals, max_vals)

cur = np.zeros((size[0], size[1], 3))
cur[:,:,0] = np.clip(X[0][:,:,0], 0, 100)
cur[:,:,1:] = ab
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.subplot(1, 2, 2)
plt.imshow(lab2rgb(cur))

## Izah

In [ ]:
from keras.layers import Conv2D, UpSampling2D, InputLayer
# Keras-dan üç fərqli layer import edirik.
#
# Conv2D:
# → 2 ölçülü konvolyusiya qatı. Şəkildə kənarları, xətləri,
#   formaları tapır (MNIST kodundakı kimi).
#
# UpSampling2D:
# → YENİ QAT. Şəklin (xüsusiyyət xəritəsinin) ölçüsünü BÖYÜDÜR.
#   MaxPooling2D-nin əksidir. Pooling kiçildirdi, bu isə böyüdür.
#
# InputLayer:
# → YENİ QAT. Modelin giriş formasını ayrıca bir qat kimi bildirir.
#   MNIST kodunda input_shape-i ilk Conv2D-nin içinə yazırdıq,
#   burada isə ayrıca qat kimi yazılır. Nəticə eynidir.


from keras.models import Sequential
# Sequential model: qatlar ardıcıl şəkildə bir-birinin arxasınca işləyir.


from skimage.color import rgb2lab, lab2rgb
# scikit-image kitabxanasından rəng fəzası çevirmə funksiyaları.
# BU KODUN ƏN VACİB HİSSƏSİ BUDUR.
#
# rgb2lab  → RGB şəkli LAB formatına çevirir
# lab2rgb  → LAB şəkli yenidən RGB formatına qaytarır
#
# LAB nədir? Rəngi başqa cür təsvir edən sistemdir.
# 3 kanalı var:
#
# L → işıqlılıq (lightness). Qiymət aralığı: 0-dan 100-ə qədər.
#     0 = qara, 100 = ağ.
#     Bu, əslində QARA-AĞ ŞƏKLİN özüdür!
#
# a → yaşıl ↔ qırmızı oxu. Təxminən -128 ... +127
# b → mavi ↔ sarı oxu.    Təxminən -128 ... +127
#
# a və b birlikdə RƏNG məlumatını daşıyır.
#
# RGB ilə müqayisə:
# RGB-də rəng və parlaqlıq bir-birinə qarışıb.
# LAB-da isə parlaqlıq (L) rəngdən (a, b) AYRILIB.
#
# Məhz buna görə rəngləndirmə məsələsi belə qurulur:
#
# Giriş:  L kanalı (qara-ağ şəkil)       → 1 kanal
# Çıxış:  a və b kanalları (rəng)        → 2 kanal
#
# Sonda L + a + b birləşdirilib rəngli şəkil alınır.


from skimage.io import imsave
# Şəkli fayla yazmaq üçün funksiya.
# DİQQƏT: Aşağıda istifadə olunmur.


import numpy as np
# NumPy — massivlərlə işləmək üçün.
#
# Bu kodda:
# - şəkli array-ə çevirmək üçün (np.array)
# - boş array yaratmaq üçün (np.zeros)
# - qiymətləri aralığa salmaq üçün (np.clip)


from google.colab import files
# Colab-da fayl yükləmək üçün modul.
# DİQQƏT: Bu kod yalnız Google Colab-da işləyir.


from io import BytesIO
# Baytları yaddaşda "virtual fayl"a çevirir (Image.open üçün lazımdır).


from PIL import Image
# Pillow — şəkli açmaq və ölçüsünü dəyişmək üçün.


import matplotlib.pyplot as plt
# Şəkilləri ekranda göstərmək üçün.


# =====================================================================
# BU KOD NƏDİR? — QARA-AĞ ŞƏKLİN AVTOMATİK RƏNGLƏNMƏSİ (colorization)
# =====================================================================
#
# Məqsəd: qara-ağ şəkli rəngli etmək.
#
# Ideya:
# 1. Rəngli şəkli LAB-a çeviririk.
# 2. L kanalını (qara-ağ hissə) modelə GİRİŞ veririk.
# 3. a və b kanallarını (rəng) modelin DOĞRU CAVABI kimi veririk.
# 4. Model "bu parlaqlıq naxışına baxanda rəng nə olmalıdır?" sualını öyrənir.
# 5. Sonra istənilən L kanalı verəndə model a, b kanallarını təxmin edir.
#
# ƏVVƏLKİ KODLARLA MÜQAYİSƏ:
#
# MNIST kodu:         modeli öyrədirdik, cavab = rəqəm (təsnifat)
# VGG16 kodu:         hazır model, öyrətmə yox, yalnız predict
# Style transfer:     model dondurulmuşdu, şəkli dəyişirdik
# BU KOD:             modeli yenə ÖYRƏDİRİK (fit), amma cavab
#                     rəqəm deyil, ŞƏKİLDİR (a və b kanalları).
#
# Yəni bu "regressiya" tipli məsələdir (ədədləri təxmin edir),
# təsnifat deyil.


upl = files.upload()
# Colab-da fayl seçmə pəncərəsi açır. Burada RƏNGLİ şəkil yükləməlisən,
# çünki model rəngləri bu şəkildən öyrənəcək.

names = list(upl.keys())
# Yüklənən faylların adlarını siyahı edir.
#
# Əvvəlki KeyError probleminin həlli budur: fayl adını əl ilə
# yazmırıq, siyahıdan götürürük.

img = Image.open(BytesIO(upl[names[0]]))
# Birinci yüklənən faylı şəkil kimi açır.
#
# DİQQƏT: Şəkil 3 kanallı (RGB) olmalıdır. Qara-ağ (grayscale)
# şəkil yükləsən, aşağıda xəta çıxar. Belə halda:
# img = img.convert('RGB')


def processed_image(img):
  # Şəkli modelə verilə biləcək formaya salan funksiya.
  # Nəticədə üç şey qaytarır: X (giriş), Y (düzgün cavab), size (ölçü).

  image = img.resize( (256, 256), Image.BILINEAR)
  # Şəkli 256 × 256 ölçüsünə gətirir.
  #
  # Image.BILINEAR → ölçü dəyişəndə piksellərin yumşaq (interpolyasiya
  # ilə) hesablanması üsuludur.
  #
  # Style transfer kodunda resize yox idi, burada var.
  # Çünki modelin sonunda ölçü 8-ə bölünməlidir (aşağıda izah olunur).

  image = np.array(image, dtype=float)
  # PIL şəklini float tipli NumPy array-ə çevirir.
  # Forma: (256, 256, 3), qiymətlər 0–255.

  size = image.shape
  # Formanı yadda saxlayır: (256, 256, 3).
  # Aşağıda reshape və sonda nəticəni yığmaq üçün lazım olacaq.

  lab = rgb2lab(1.0/255*image)
  # RGB → LAB çevirməsi.
  #
  # 1.0/255*image → əvvəlcə qiymətləri 0–1 aralığına gətirir
  # (rgb2lab bunu tələb edir, 0–255 yox).
  #
  # Nəticə: (256, 256, 3) forma, amma artıq LAB kanalları:
  # [:,:,0] → L (0 ... 100)
  # [:,:,1] → a (təxminən -128 ... 127)
  # [:,:,2] → b (təxminən -128 ... 127)

  X, Y = lab[:,:,0], lab[:,:,1:]
  # LAB-ı iki hissəyə bölür:
  #
  # X → yalnız L kanalı      → forma (256, 256)    → modelin GİRİŞİ
  # Y → a və b kanalları     → forma (256, 256, 2) → düzgün CAVAB
  #
  # [:,:,1:] → ikinci kanaldan sona qədər, yəni a və b.

  Y /= 128    # нормируем выходные значение в диапазон от -1 до 1
  # Yuxarıdakı rus dilindəki şərh belə tərcümə olunur:
  # "çıxış qiymətlərini -1-dən 1-ə qədər aralığa normallaşdırırıq".
  #
  # a və b -128 ... 127 aralığındadır. 128-ə bölünəndə
  # təxminən -1 ... +1 olur.
  #
  # Niyə? Modelin son qatında tanh aktivasiyası var (aşağıda),
  # tanh isə yalnız -1 ilə +1 arasında qiymət verə bilir.
  # Cavab da həmin aralıqda olmalıdır ki, model ona çata bilsin.

  X = X.reshape(1, size[0], size[1], 1)
  # L kanalının formasını dəyişir:
  #
  # (256, 256)  →  (1, 256, 256, 1)
  #
  # 1   → batch ölçüsü (1 şəkil)
  # 256 → hündürlük
  # 256 → en
  # 1   → kanal sayı (yalnız L)
  #
  # MNIST kodunda bunu np.expand_dims ilə edirdik,
  # burada reshape ilə edilir. Nəticə eynidir.
  #
  # Qeyd: X-in qiymətləri 0–100 aralığındadır, 0–1 deyil.
  # Yəni giriş normallaşdırılmayıb. İşləyir, amma X/100 etsək
  # təlim daha stabil ola bilərdi.

  Y = Y.reshape(1, size[0], size[1], 2)
  # a və b kanallarının forması: (256, 256, 2) → (1, 256, 256, 2)

  return X, Y, size
  # Giriş, cavab və ölçünü qaytarır.


X, Y, size = processed_image(img)
# Yüklənmiş şəkli hazırlayırıq.
#
# X → (1, 256, 256, 1)  qara-ağ hissə (giriş)
# Y → (1, 256, 256, 2)  rəng hissəsi (düzgün cavab)


# =====================================================================
# MODELİN QURULMASI
# =====================================================================
#
# Model iki hissədən ibarətdir:
#
# 1. KİÇİLƏN hissə (encoder): şəkli kiçildir, xüsusiyyətləri çıxarır.
#    256 × 256  →  128 × 128  →  64 × 64  →  32 × 32
#
# 2. BÖYÜYƏN hissə (decoder): kiçik xüsusiyyətlərdən yenidən
#    böyük şəkil (rəng xəritəsi) yaradır.
#    32 × 32  →  64 × 64  →  128 × 128  →  256 × 256
#
# Yəni forma "qum saatı" kimidir: əvvəl daralır, sonra genişlənir.

model = Sequential()
# Boş ardıcıl model yaradır.

model.add(InputLayer(input_shape=(None, None, 1)))
# Giriş qatı.
#
# (None, None, 1):
# → None, None → hündürlük və en İSTƏNİLƏN ola bilər.
# → 1 → kanal sayı (yalnız L).
#
# Bu, modelin müxtəlif ölçülü şəkillər qəbul edə bilməsi deməkdir.
# Çünki bu modeldə Dense qat yoxdur, yalnız Conv və UpSampling var.
# (MNIST kodunda Dense olduğu üçün ölçü sabit idi.)

model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
# 64 filterlik konvolyusiya. padding='same' → ölçü dəyişmir.
# (256, 256, 1)  →  (256, 256, 64)
# Parametr: 3×3×1×64 + 64 = 640

model.add(Conv2D(64, (3, 3), activation='relu', padding='same', strides=2))
# strides=2 → filter hər dəfə 2 piksel sürüşür.
# Nəticədə ölçü 2 dəfə KİÇİLİR.
# (256, 256, 64)  →  (128, 128, 64)
#
# Bu, MaxPooling-in əvəzidir. MNIST kodunda ölçünü MaxPooling2D
# kiçildirdi, burada isə strides=2 olan Conv2D edir.
# Fərq: pooling heç nə öyrənmir, strided Conv isə öyrənir.

model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
# Filter sayı artır: 128. Ölçü eyni.
# (128, 128, 64)  →  (128, 128, 128)

model.add(Conv2D(128, (3, 3), activation='relu', padding='same', strides=2))
# Yenə ölçü kiçilir.
# (128, 128, 128)  →  (64, 64, 128)

model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
# (64, 64, 128)  →  (64, 64, 256)

model.add(Conv2D(256, (3, 3), activation='relu', padding='same', strides=2))
# Üçüncü və axırıncı kiçilmə.
# (64, 64, 256)  →  (32, 32, 256)

model.add(Conv2D(512, (3, 3), activation='relu', padding='same'))
# Şəbəkənin ən "dərin" nöqtəsi. 512 filter.
# (32, 32, 256)  →  (32, 32, 512)
#
# Burada şəkil ən kiçik ölçüdədir (32 × 32), amma ən çox xüsusiyyət
# xəritəsi var. Bu qat şəkildəki yüksək səviyyəli mənanı "anlayır":
# bu səmadır, bu ağacdır, bu insan üzüdür.
# Rəngi təxmin etmək üçün bu məna vacibdir (səma mavi, ot yaşıl).

model.add(Conv2D(256, (3, 3), activation='relu', padding='same'))
# Filter sayı yenidən azalmağa başlayır (decoder başlayır).
# (32, 32, 512)  →  (32, 32, 256)

model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
# (32, 32, 256)  →  (32, 32, 128)

model.add(UpSampling2D((2, 2)))
# Ölçünü 2 dəfə BÖYÜDÜR.
# (32, 32, 128)  →  (64, 64, 128)
#
# Necə işləyir? Hər pikseli 2 × 2 bölgəyə təkrarlayır:
#
# [1, 2]        [1, 1, 2, 2]
# [3, 4]   →    [1, 1, 2, 2]
#               [3, 3, 4, 4]
#               [3, 3, 4, 4]
#
# Heç bir öyrədilən parametri yoxdur (0 parametr).
# MaxPooling-in tərsi kimi düşün.

model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
# Böyüdülmüş xəritəni işləyir, detalları düzəldir.
# (64, 64, 128)  →  (64, 64, 64)

model.add(UpSampling2D((2, 2)))
# Yenə 2 dəfə böyüdür.
# (64, 64, 64)  →  (128, 128, 64)

model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
# (128, 128, 64)  →  (128, 128, 32)

model.add(Conv2D(2, (3, 3), activation='tanh', padding='same'))
# SON KONVOLYUSİYA QATI. Nəticəni verən qat budur.
#
# 2 filter → 2 kanal → a və b kanalları!
# Hər filter bir rəng kanalını təxmin edir.
#
# activation='tanh':
# → çıxışı -1 ilə +1 arasına salır.
# → Biz Y-ni 128-ə böləndə qiymətləri məhz bu aralığa gətirmişdik.
#   Yəni model və cavab eyni aralıqdadır.
#
# (128, 128, 32)  →  (128, 128, 2)
#
# Burada softmax yoxdur, çünki rəqəm sinfi seçmirik,
# ədədi qiymət təxmin edirik.

model.add(UpSampling2D((2, 2)))
# Sonuncu böyütmə.
# (128, 128, 2)  →  (256, 256, 2)
#
# İndi çıxışın forması girişin ölçüsü ilə eynidir (256 × 256),
# yalnız kanal sayı 1 yox, 2-dir.
#
# Ölçü uyğunluğu: 3 dəfə kiçildik (÷2 ÷2 ÷2 = ÷8),
# 3 dəfə böyüdük (×2 ×2 ×2 = ×8). Beləliklə ölçü qayıdır.
# Buna görə şəklin ölçüsü 8-ə bölünməlidir (256 bölünür).
#
# Ümumi parametr sayı: təxminən 3.9 milyon.


model.compile(optimizer='adam', loss='mse')
# Öyrətmənin parametrlərini təyin edir.
#
# optimizer='adam' → MNIST kodundakı kimi.
#
# loss='mse' → Mean Squared Error (orta kvadratik xəta).
#   Modelin təxmin etdiyi a, b qiymətləri ilə düzgün a, b qiymətləri
#   arasındakı fərqin kvadratının ortası.
#
# MNIST-də loss='categorical_crossentropy' idi (təsnifat üçün).
# Burada ədədlərin yaxınlığını ölçürük, ona görə MSE.
#
# metrics=['accuracy'] YOXDUR, çünki accuracy təsnifat üçündür.
# Rəng təxmin edəndə "düzgün/yanlış" demək mənasızdır.


model.fit(x=X, y=Y, batch_size=1, epochs=50)
# Öyrətməni başladır.
#
# x=X → giriş: qara-ağ hissə (L)
# y=Y → düzgün cavab: rəng hissəsi (a, b)
#
# batch_size=1, epochs=50
# → CƏMİ BİR şəkil var, ona görə hər epoch-da 1 addım olur.
#   Yəni model cəmi 50 dəfə yenilənir.
#
# ƏN VACİB QEYD:
# Model yalnız BİR şəkil üzərində öyrədilir. Yəni o, ümumi
# "rənglənməni" yox, məhz BU şəklin rənglərini əzbərləyir
# (overfitting). Bu, tələbə kimi bir sualın cavabını əzbərləmək kimidir.
#
# Nəticə:
# - Eyni şəkli (qara-ağ versiyasını) versən → yaxşı rəngləyər.
# - Tamam başqa şəkil versən → rənglər zəif və qeyri-dəqiq olar.
#
# Real rəngləndirmə sistemləri minlərlə şəkil üzərində öyrədilir.
# validation_split də yoxdur, çünki yoxlayacaq ikinci şəkil yoxdur.


upl = files.upload()
names = list(upl.keys())
img = Image.open(BytesIO(upl[names[0]]))
# İkinci dəfə fayl yükləmə pəncərəsi açılır.
# Bu dəfə RƏNGLƏNDİRİLƏCƏK şəkli yükləyirsən.
#
# Əgər birinci şəklin özünü yükləsən → model onu yaxşı rəngləyəcək.
# Başqa şəkil yükləsən → nəticə zəif olacaq (yuxarıdakı səbəbdən).
#
# Qeyd: Bu şəkil rəngli ola bilər. Aşağıda yalnız L kanalı
# götürüləcək, yəni rəngi atılacaq.

X, Y, size = processed_image(img)
# Yeni şəkli eyni funksiya ilə hazırlayırıq.
#
# Burada Y də hesablanır, amma aşağıda istifadə olunmur.
# Yalnız X (qara-ağ hissə) lazımdır.


output = model.predict(X)
# Model qara-ağ hissəyə (L) baxıb a və b kanallarını təxmin edir.
#
# X → (1, 256, 256, 1)
# output → (1, 256, 256, 2), qiymətlər -1 ... +1
#
# Bu predict-dir, öyrətmə yox. Model heç nə öyrənmir.


output *= 128
# Qiymətləri geri böyüdür: -1...+1 → təxminən -128...+128.
# Bu, öyrətmədən əvvəl etdiyimiz Y /= 128 əməliyyatının tərsidir.
#
# Eyni ideya: style transfer-də deprocess_img preprocess-in tərsini edirdi.

min_vals, max_vals = -128, 127
ab = np.clip(output[0], min_vals, max_vals)
# output[0] → batch ölçüsünü silir: (1,256,256,2) → (256,256,2)
# np.clip → qiymətləri -128 ... 127 aralığında saxlayır.
# (LAB-da a və b-nin icazə verilən aralığı budur.)


cur = np.zeros((size[0], size[1], 3))
# Boş LAB şəkli yaradır: (256, 256, 3), hamısı sıfır.
# İndi onu doldururuq.

cur[:,:,0] = np.clip(X[0][:,:,0], 0, 100)
# Birinci kanal (L) = ORİJİNAL qara-ağ hissə.
#
# X[0][:,:,0] → batch və kanal ölçüsünü atır: (256, 256)
# np.clip(..., 0, 100) → L-in icazə verilən aralığı 0–100-dür.
#
# Vacib: parlaqlıq (L) modelin təxmini deyil, orijinal şəkildən
# gəlir. Model yalnız rəngi (a, b) əlavə edir.
# Buna görə şəklin detalları və kəskinliyi qorunur.

cur[:,:,1:] = ab
# İkinci və üçüncü kanallar (a, b) = modelin təxmin etdiyi rənglər.
#
# İndi cur tam LAB şəklidir:
# L → orijinal    a, b → modelin təxmini


plt.subplot(1, 2, 1)
plt.imshow(img)
# Solda yüklədiyin orijinal şəkli göstərir.
# (Bu, resize olunmamış orijinaldır.)

plt.subplot(1, 2, 2)
plt.imshow(lab2rgb(cur))
# Sağda rənglənmiş nəticəni göstərir.
#
# lab2rgb(cur) → LAB-ı yenidən RGB-yə çevirir, çünki
# ekran LAB yox, RGB göstərir.